# Ir Além — Detecção em tempo real com webcam do PC

Este notebook é independente das Entregas 1 e 2. Ele executa a aplicação local que abre a webcam, usa o `best.pt` produzido na Entrega 1 para detectar `chair` e `bottle` e registra evidências reais.

**Objetivo:** comprovar coleta, transmissão e inferência ao vivo sem usar ESP32-CAM. O iPhone transmite pelo Iriun Webcam via Wi-Fi para o Windows, que o expõe como webcam virtual ao Python.

## 1. Pré-requisitos

Execute este notebook localmente (Jupyter ou VS Code), pois o Colab não acessa a câmera física do computador. Use Python 3.10 ou superior, permita o acesso à câmera no Windows e feche aplicativos que já estejam usando a webcam.

In [ ]:
from pathlib import Path
import sys

# Localiza a raiz do repositório, mesmo se o notebook for aberto pelo VS Code/Jupyter.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'runs').is_dir():
    ROOT = ROOT.parent

WEIGHTS = ROOT / 'runs' / 'treino_60ep' / 'weights' / 'best.pt'
APP = ROOT / 'ir_alem_webcam' / 'src' / 'webcam_yolo.py'
REQ = ROOT / 'ir_alem_webcam' / 'requirements.txt'
assert WEIGHTS.is_file(), f'Peso não encontrado: {WEIGHTS}'
assert APP.is_file(), f'Aplicação não encontrada: {APP}'
print('Raiz:', ROOT)
print('Peso:', WEIGHTS)

## 2. Dependências

Rode a próxima célula uma única vez no ambiente Python escolhido. Na primeira inferência, o PyTorch pode baixar para o cache o código oficial do YOLOv5; mantenha conexão com a internet nessa etapa.

In [ ]:
# Descomente somente se as dependências ainda não estiverem instaladas.
# !{sys.executable} -m pip install -r {REQ}

## 3. Demonstração ao vivo

A célula abaixo abre uma janela do OpenCV. Posicione uma cadeira e/ou garrafa em frente à webcam. Pressione **S** para salvar uma evidência e **Q** (ou Esc) para encerrar. A saída só termina quando a janela for fechada; esse é o comportamento esperado para uma demonstração ao vivo.

In [ ]:
import subprocess

cmd = [
    sys.executable, str(APP),
    '--weights', str(WEIGHTS),
    '--camera', '0',
    '--conf', '0.25',
]
print('Comando:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 4. Verificação das evidências

Cada pressionamento de **S** cria `raw_*.jpg`, `detected_*.jpg` e `capture_*.json`; o arquivo `detections.csv` concentra todas as caixas detectadas. As imagens anotadas devem ser incluídas no GitHub como comprovação do resultado, depois de uma execução real.

In [ ]:
import json
from IPython.display import display, Image

EVIDENCIAS = ROOT / 'ir_alem_webcam' / 'evidencias'
capturas = sorted(EVIDENCIAS.glob('capture_*.json'))
print(f'Capturas registradas: {len(capturas)}')
for metadado in capturas[-3:]:
    dados = json.loads(metadado.read_text(encoding='utf-8'))
    print(f"{metadado.name}: {len(dados['detections'])} detecção(ões), {dados['estimated_fps']} FPS")
    display(Image(filename=str(EVIDENCIAS / dados['annotated_image'])))

## Conclusão crítica

A demonstração valida o caminho completo iPhone → Wi-Fi/Iriun → webcam virtual no Windows → Python/OpenCV → YOLOv5 customizado → resultado visual e evidência. O modelo é o mesmo da Entrega 1, portanto não há novo treinamento. O Iriun simplifica a prova de conceito e evita custo de hardware; uma solução com ESP32-CAM exigiria firmware próprio, limitações de energia e processamento em um computador/servidor receptor. O resultado observado deve ser discutido considerando iluminação, distância, ângulo, estabilidade da rede e limitações do dataset pequeno.